# I. Fixed Effects

# 3 Group Fixed Effects

# 100 Group Fixed Effects

# Overall R-Squared vs. Within Groups R-Squared

# Fixed Effects with Dummy Variables

# Two-Way Fixed Effects

# II. Regression Topics

# Heteroskedasticity

# Regression with Autocorrelated Data

# Log Transformation

# The Big Question: When can we compare observations within the same group to remove unobserved group-level confounding?

## Why fixed effects matter

## In causal analysis, groups often differ in ways we cannot fully observe. Cities differ in infrastructure, hospitals differ in patient populations, stores differ in neighborhood income, and users differ in baseline engagement.

## If those baseline differences are correlated with X, a pooled regression may confuse “group differences” with the effect of X. Fixed effects ask a narrower question: within the same group, when X changes, how does Y change?

## Case study: If coffee shops increase in a city, that city doesn't become Seattle.  It becomes more like Seattle in number of coffee shops, but it doesn't become Seattle in other ways. Seattle also has a tech-oriented economy and is near the mountains, for instance.  Thus, we have to "control for the city."  We could look at how different neighborhoods with varying numbers of coffee shops behave within each city.

## Prerequisite reminder: what does an intercept do?

## In a simple regression line, the slope controls how Y changes with X. The intercept controls the baseline level of Y when X = 0.

## Fixed effects are mostly about giving each group its own intercept while keeping one common slope.

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from IPython.display import display, HTML


display(HTML("""
<div style="font-size: 28px; line-height: 1.45; max-width: 1100px;">

<h1 style="font-size: 42px;">Visual Intuition: Fixed Effects as Demeaning</h1>

<p>
A fixed-effects regression estimates the relationship between <b>X</b> and <b>Y</b>
after removing each group's baseline level.
</p>

<p>
The key operation is:
</p>

<div style="background-color:#eaf4ff; padding: 18px; margin: 12px 0;">
<b>within-group value = individual value − group average</b>
</div>

<p>
So instead of asking:
</p>

<div style="background-color:#fff3cd; padding: 18px; margin: 12px 0;">
Do groups with higher average X also have higher average Y?
</div>

<p>
fixed effects ask:
</p>

<div style="background-color:#d4edda; padding: 18px; margin: 12px 0;">
Within the same group, when X is above that group's usual X, is Y also above that group's usual Y?
</div>

</div>
"""))


def make_three_group_data(
    n_per_group=12,
    beta=2.0,
    fixed_effect_gap=5.0,
    x_mean_gap=3.0,
    noise_sd=1.0,
    seed=123,
):
    rng = np.random.default_rng(seed)

    groups = ["A", "B", "C"]

    x_means = {
        "A": -x_mean_gap,
        "B": 0.0,
        "C": x_mean_gap,
    }

    fixed_effects = {
        "A": -fixed_effect_gap,
        "B": 0.0,
        "C": fixed_effect_gap,
    }

    rows = []

    for g in groups:
        x = rng.normal(loc=x_means[g], scale=1.0, size=n_per_group)
        y = fixed_effects[g] + beta * x + rng.normal(0.0, noise_sd, size=n_per_group)

        for i in range(n_per_group):
            rows.append(
                {
                    "group": g,
                    "x": x[i],
                    "y": y[i],
                }
            )

    df = pd.DataFrame(rows)

    df["xbar_g"] = df.groupby("group")["x"].transform("mean")
    df["ybar_g"] = df.groupby("group")["y"].transform("mean")

    df["x_demeaned"] = df["x"] - df["xbar_g"]
    df["y_demeaned"] = df["y"] - df["ybar_g"]

    return df


def ols_slope_intercept(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    xbar = np.mean(x)
    ybar = np.mean(y)

    sxx = np.sum((x - xbar) ** 2)

    if sxx <= 1e-12:
        return np.nan, np.nan

    beta = np.sum((x - xbar) * (y - ybar)) / sxx
    intercept = ybar - beta * xbar

    return beta, intercept


def draw_demeaning_demo(
    n_per_group=12,
    beta=2.0,
    fixed_effect_gap=5.0,
    x_mean_gap=3.0,
    noise_sd=1.0,
    show_arrows=True,
    show_lines=True,
    seed=123,
):
    df = make_three_group_data(
        n_per_group=n_per_group,
        beta=beta,
        fixed_effect_gap=fixed_effect_gap,
        x_mean_gap=x_mean_gap,
        noise_sd=noise_sd,
        seed=seed,
    )

    colors = {
        "A": "tab:blue",
        "B": "tab:orange",
        "C": "tab:green",
    }

    pooled_beta, pooled_intercept = ols_slope_intercept(df["x"], df["y"])
    within_beta, within_intercept = ols_slope_intercept(
        df["x_demeaned"],
        df["y_demeaned"],
    )

    group_means = df.groupby("group")[["x", "y"]].mean()

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    ax = axes[0]

    for g in ["A", "B", "C"]:
        sub = df[df["group"] == g]

        ax.scatter(
            sub["x"],
            sub["y"],
            color=colors[g],
            s=75,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.5,
            label=f"Group {g}",
            zorder=3,
        )

        ax.scatter(
            group_means.loc[g, "x"],
            group_means.loc[g, "y"],
            color=colors[g],
            s=340,
            marker="X",
            edgecolor="black",
            linewidth=1.8,
            label=f"Mean {g}",
            zorder=5,
        )

        if show_arrows:
            for _, row in sub.iterrows():
                ax.annotate(
                    "",
                    xy=(row["xbar_g"], row["ybar_g"]),
                    xytext=(row["x"], row["y"]),
                    arrowprops=dict(
                        arrowstyle="->",
                        color="black",
                        alpha=0.55,
                        linewidth=1.7,
                        shrinkA=5,
                        shrinkB=8,
                        mutation_scale=13,
                    ),
                    zorder=2,
                )

    x_grid = np.linspace(df["x"].min() - 1.0, df["x"].max() + 1.0, 200)

    if show_lines:
        ax.plot(
            x_grid,
            pooled_intercept + pooled_beta * x_grid,
            color="black",
            linewidth=3,
            label=f"Pooled slope = {pooled_beta:.2f}",
            zorder=4,
        )

        for g in ["A", "B", "C"]:
            group_intercept = group_means.loc[g, "y"] - within_beta * group_means.loc[g, "x"]

            ax.plot(
                x_grid,
                group_intercept + within_beta * x_grid,
                color=colors[g],
                linewidth=2.5,
                linestyle="--",
                label=f"FE line {g}",
                zorder=4,
            )

    ax.set_title("Original Data: Arrows Point Toward Group Means", fontsize=17)
    ax.set_xlabel("X", fontsize=14)
    ax.set_ylabel("Y", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc="best")

    ax = axes[1]

    for g in ["A", "B", "C"]:
        sub = df[df["group"] == g]

        ax.scatter(
            sub["x_demeaned"],
            sub["y_demeaned"],
            color=colors[g],
            s=75,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.5,
            label=f"Group {g}",
            zorder=3,
        )

    x_dm_grid = np.linspace(
        df["x_demeaned"].min() - 0.5,
        df["x_demeaned"].max() + 0.5,
        200,
    )

    if show_lines:
        ax.plot(
            x_dm_grid,
            within_intercept + within_beta * x_dm_grid,
            color="black",
            linewidth=3,
            label=f"Within/FE slope = {within_beta:.2f}",
            zorder=4,
        )

    ax.axhline(0, color="gray", linewidth=1.5, alpha=0.7)
    ax.axvline(0, color="gray", linewidth=1.5, alpha=0.7)

    ax.set_title("Demeaned Data: Group Means Have Become Zero", fontsize=17)
    ax.set_xlabel("X minus group mean of X", fontsize=14)
    ax.set_ylabel("Y minus group mean of Y", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10, loc="best")

    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame(
        {
            "quantity": [
                "True within-group slope used to generate data",
                "Naive pooled slope",
                "Fixed-effects / within slope",
            ],
            "value": [
                beta,
                pooled_beta,
                within_beta,
            ],
        }
    )

    display(summary)

    display(HTML(f"""
    <div style="font-size: 24px; line-height: 1.45; max-width: 1100px;">

    <h2 style="font-size: 34px;">What the arrows mean</h2>

    <p>
    Each arrow points from an observation to its own group mean.
    This shows the baseline that will be subtracted away.
    </p>

    <div style="background-color:#eaf4ff; padding: 18px; margin: 12px 0;">
    For each point, fixed effects use:
    <br><br>
    <b>X minus group average X</b>
    <br>
    and
    <br>
    <b>Y minus group average Y</b>
    </div>

    <p>
    The right plot shows those deviations after the group means have been moved to the origin.
    </p>

    <div style="background-color:#fff3cd; padding: 18px; margin-top: 12px;">
    <b>Main lesson:</b> fixed effects estimate the relationship using within-group variation,
    not mainly from differences between group averages.
    </div>

    </div>
    """))


interact(
    draw_demeaning_demo,
    n_per_group=IntSlider(
        value=12,
        min=4,
        max=60,
        step=1,
        description="N/group",
        continuous_update=False,
    ),
    beta=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="True beta",
        continuous_update=False,
    ),
    fixed_effect_gap=FloatSlider(
        value=5.0,
        min=-10.0,
        max=10.0,
        step=0.5,
        description="FE gap",
        continuous_update=False,
    ),
    x_mean_gap=FloatSlider(
        value=3.0,
        min=-8.0,
        max=8.0,
        step=0.5,
        description="X mean gap",
        continuous_update=False,
    ),
    noise_sd=FloatSlider(
        value=1.0,
        min=0.0,
        max=8.0,
        step=0.25,
        description="Noise SD",
        continuous_update=False,
    ),
    show_arrows=Checkbox(
        value=True,
        description="Show arrows",
    ),
    show_lines=Checkbox(
        value=True,
        description="Show lines",
    ),
    seed=IntSlider(
        value=123,
        min=1,
        max=9999,
        step=1,
        description="Seed",
        continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=12, continuous_update=False, description='N/group', max=60, min=4), Floa…

# Fixed Effects

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Dropdown
from IPython.display import display


def make_group_data(
    group_name,
    n,
    x_mean,
    x_variance,
    fixed_effect,
    beta1_true,
    y_noise_sd,
    rng,
):
    x = rng.normal(loc=x_mean, scale=np.sqrt(x_variance), size=n)
    y = fixed_effect + beta1_true * x + rng.normal(0.0, y_noise_sd, size=n)

    return pd.DataFrame(
        {
            "group": group_name,
            "x": x,
            "y": y,
        }
    )


def make_dataset(
    n_A,
    n_B,
    n_C,
    mean_x_A,
    mean_x_B,
    mean_x_C,
    var_x_A,
    var_x_B,
    var_x_C,
    fixed_A,
    fixed_B,
    fixed_C,
    beta_A,
    beta_B,
    beta_C,
    y_noise_sd,
    rng,
):
    df_A = make_group_data(
        "A", n_A, mean_x_A, var_x_A, fixed_A, beta_A, y_noise_sd, rng
    )
    df_B = make_group_data(
        "B", n_B, mean_x_B, var_x_B, fixed_B, beta_B, y_noise_sd, rng
    )
    df_C = make_group_data(
        "C", n_C, mean_x_C, var_x_C, fixed_C, beta_C, y_noise_sd, rng
    )

    return pd.concat([df_A, df_B, df_C], ignore_index=True)


def ols_slope_intercept(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    xbar = np.mean(x)
    ybar = np.mean(y)

    sxx = np.sum((x - xbar) ** 2)

    if sxx <= 1e-12:
        return np.nan, np.nan

    beta = np.sum((x - xbar) * (y - ybar)) / sxx
    intercept = ybar - beta * xbar

    return beta, intercept


def fit_pooled(df):
    beta, intercept = ols_slope_intercept(df["x"], df["y"])

    return {
        "intercept": intercept,
        "beta": beta,
    }


def fit_fixed_effects(df):
    work = df.copy()

    work["xbar_g"] = work.groupby("group")["x"].transform("mean")
    work["ybar_g"] = work.groupby("group")["y"].transform("mean")

    work["x_dm"] = work["x"] - work["xbar_g"]
    work["y_dm"] = work["y"] - work["ybar_g"]

    beta = np.sum(work["x_dm"] * work["y_dm"]) / np.sum(work["x_dm"] ** 2)

    group_means = work.groupby("group")[["x", "y"]].mean()

    intercepts = {}

    for g in ["A", "B", "C"]:
        intercepts[g] = group_means.loc[g, "y"] - beta * group_means.loc[g, "x"]

    return {
        "beta": beta,
        "intercepts": intercepts,
    }


def fit_separate(df):
    results = {}

    for g in ["A", "B", "C"]:
        sub = df[df["group"] == g]
        beta, intercept = ols_slope_intercept(sub["x"], sub["y"])

        results[g] = {
            "beta": beta,
            "intercept": intercept,
            "n": len(sub),
        }

    return results


def predict_pooled(model, df):
    return model["intercept"] + model["beta"] * df["x"].to_numpy()


def predict_fixed_effects(model, df):
    preds = np.zeros(len(df))

    for idx, row in df.reset_index(drop=True).iterrows():
        g = row["group"]
        preds[idx] = model["intercepts"][g] + model["beta"] * row["x"]

    return preds


def predict_separate(model, df):
    preds = np.zeros(len(df))

    for idx, row in df.reset_index(drop=True).iterrows():
        g = row["group"]
        preds[idx] = model[g]["intercept"] + model[g]["beta"] * row["x"]

    return preds


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


def run_one_experiment(
    n_A,
    n_B,
    n_C,
    n_test_per_group,
    mean_x_A,
    mean_x_B,
    mean_x_C,
    var_x_A,
    var_x_B,
    var_x_C,
    fixed_A,
    fixed_B,
    fixed_C,
    beta_A,
    beta_B,
    beta_C,
    y_noise_sd,
    rng,
):
    train = make_dataset(
        n_A,
        n_B,
        n_C,
        mean_x_A,
        mean_x_B,
        mean_x_C,
        var_x_A,
        var_x_B,
        var_x_C,
        fixed_A,
        fixed_B,
        fixed_C,
        beta_A,
        beta_B,
        beta_C,
        y_noise_sd,
        rng,
    )

    test = make_dataset(
        n_test_per_group,
        n_test_per_group,
        n_test_per_group,
        mean_x_A,
        mean_x_B,
        mean_x_C,
        var_x_A,
        var_x_B,
        var_x_C,
        fixed_A,
        fixed_B,
        fixed_C,
        beta_A,
        beta_B,
        beta_C,
        y_noise_sd,
        rng,
    )

    pooled = fit_pooled(train)
    fe = fit_fixed_effects(train)
    sep = fit_separate(train)

    pooled_pred = predict_pooled(pooled, test)
    fe_pred = predict_fixed_effects(fe, test)
    sep_pred = predict_separate(sep, test)

    n_total = n_A + n_B + n_C

    sep_weighted_beta_by_n = (
        sep["A"]["beta"] * n_A
        + sep["B"]["beta"] * n_B
        + sep["C"]["beta"] * n_C
    ) / n_total

    sxx_weights = {}

    for g in ["A", "B", "C"]:
        sub = train[train["group"] == g]
        x = sub["x"].to_numpy()
        sxx_weights[g] = np.sum((x - np.mean(x)) ** 2)

    total_sxx = sxx_weights["A"] + sxx_weights["B"] + sxx_weights["C"]

    sep_weighted_beta_by_sxx = (
        sep["A"]["beta"] * sxx_weights["A"]
        + sep["B"]["beta"] * sxx_weights["B"]
        + sep["C"]["beta"] * sxx_weights["C"]
    ) / total_sxx

    return {
        "train": train,
        "test": test,
        "pooled": pooled,
        "fe": fe,
        "sep": sep,
        "pooled_test_rmse": rmse(test["y"], pooled_pred),
        "fe_test_rmse": rmse(test["y"], fe_pred),
        "sep_test_rmse": rmse(test["y"], sep_pred),
        "sep_weighted_beta_by_n": sep_weighted_beta_by_n,
        "sep_weighted_beta_by_sxx": sep_weighted_beta_by_sxx,
    }


def run_demo(
    scenario="Common slope",
    mean_x_A=-3.0,
    mean_x_B=0.0,
    mean_x_C=3.0,
    var_x_A=1.0,
    var_x_B=1.0,
    var_x_C=1.0,
    fixed_A=-4.0,
    fixed_B=0.0,
    fixed_C=4.0,
    beta_A=2.0,
    beta_B=2.0,
    beta_C=2.0,
    n_A=10,
    n_B=10,
    n_C=10,
    n_test_per_group=500,
    y_noise_sd=2.0,
    n_sims=500,
    show_lines=True,
    seed=123,
):
    if scenario == "Common slope":
        beta_B = beta_A
        beta_C = beta_A

    true_common_beta = (beta_A + beta_B + beta_C) / 3

    rng = np.random.default_rng(seed)

    one = run_one_experiment(
        n_A=n_A,
        n_B=n_B,
        n_C=n_C,
        n_test_per_group=n_test_per_group,
        mean_x_A=mean_x_A,
        mean_x_B=mean_x_B,
        mean_x_C=mean_x_C,
        var_x_A=var_x_A,
        var_x_B=var_x_B,
        var_x_C=var_x_C,
        fixed_A=fixed_A,
        fixed_B=fixed_B,
        fixed_C=fixed_C,
        beta_A=beta_A,
        beta_B=beta_B,
        beta_C=beta_C,
        y_noise_sd=y_noise_sd,
        rng=rng,
    )

    train = one["train"]
    pooled = one["pooled"]
    fe = one["fe"]
    sep = one["sep"]

    group_colors = {
        "A": "tab:blue",
        "B": "tab:orange",
        "C": "tab:green",
    }

    fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharex=True, sharey=True)

    x_min = train["x"].min()
    x_max = train["x"].max()
    x_margin = 0.2 * max(x_max - x_min, 1.0)
    x_grid = np.linspace(x_min - x_margin, x_max + x_margin, 300)

    ax = axes[0]

    for g in ["A", "B", "C"]:
        sub = train[train["group"] == g]

        ax.scatter(
            sub["x"],
            sub["y"],
            color=group_colors[g],
            alpha=0.8,
            label=f"Group {g}",
        )

    if show_lines:
        ax.plot(
            x_grid,
            pooled["intercept"] + pooled["beta"] * x_grid,
            color="black",
            linewidth=3,
            label="Pooled line",
        )

    ax.set_title("Naive Pooled Regression")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[1]

    for g in ["A", "B", "C"]:
        sub = train[train["group"] == g]

        ax.scatter(
            sub["x"],
            sub["y"],
            color=group_colors[g],
            alpha=0.8,
            label=f"Group {g}",
        )

        if show_lines:
            ax.plot(
                x_grid,
                fe["intercepts"][g] + fe["beta"] * x_grid,
                color=group_colors[g],
                linewidth=2.5,
                label=f"FE line {g}",
            )

    ax.set_title("Fixed Effects\nGroup Intercepts, Common Slope")
    ax.set_xlabel("X")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[2]

    for g in ["A", "B", "C"]:
        sub = train[train["group"] == g]

        ax.scatter(
            sub["x"],
            sub["y"],
            color=group_colors[g],
            alpha=0.8,
            label=f"Group {g}",
        )

        if show_lines:
            ax.plot(
                x_grid,
                sep[g]["intercept"] + sep[g]["beta"] * x_grid,
                color=group_colors[g],
                linewidth=2.5,
                linestyle="--",
                label=f"Separate line {g}",
            )

    ax.set_title("Separate Regressions\nGroup Intercepts, Group Slopes")
    ax.set_xlabel("X")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

    one_run_table = pd.DataFrame(
        {
            "model_or_quantity": [
                "Naive pooled beta",
                "Fixed effects beta",
                "Separate beta A",
                "Separate beta B",
                "Separate beta C",
                "Separate weighted beta by n",
                "Separate weighted beta by within-group Sxx",
            ],
            "estimate": [
                pooled["beta"],
                fe["beta"],
                sep["A"]["beta"],
                sep["B"]["beta"],
                sep["C"]["beta"],
                one["sep_weighted_beta_by_n"],
                one["sep_weighted_beta_by_sxx"],
            ],
        }
    )

    print("One Random Training Sample")
    print("--------------------------")
    display(one_run_table)

    intercept_table = pd.DataFrame(
        {
            "group": ["A", "B", "C"],
            "true_intercept": [fixed_A, fixed_B, fixed_C],
            "FE_intercept": [
                fe["intercepts"]["A"],
                fe["intercepts"]["B"],
                fe["intercepts"]["C"],
            ],
            "separate_intercept": [
                sep["A"]["intercept"],
                sep["B"]["intercept"],
                sep["C"]["intercept"],
            ],
            "true_beta": [beta_A, beta_B, beta_C],
            "separate_beta": [
                sep["A"]["beta"],
                sep["B"]["beta"],
                sep["C"]["beta"],
            ],
        }
    )

    display(intercept_table)

    sim_rows = []

    rng_mc = np.random.default_rng(seed + 100000)

    for sim in range(n_sims):
        result = run_one_experiment(
            n_A=n_A,
            n_B=n_B,
            n_C=n_C,
            n_test_per_group=n_test_per_group,
            mean_x_A=mean_x_A,
            mean_x_B=mean_x_B,
            mean_x_C=mean_x_C,
            var_x_A=var_x_A,
            var_x_B=var_x_B,
            var_x_C=var_x_C,
            fixed_A=fixed_A,
            fixed_B=fixed_B,
            fixed_C=fixed_C,
            beta_A=beta_A,
            beta_B=beta_B,
            beta_C=beta_C,
            y_noise_sd=y_noise_sd,
            rng=rng_mc,
        )

        sim_rows.append(
            {
                "pooled_beta": result["pooled"]["beta"],
                "fe_beta": result["fe"]["beta"],
                "sep_weighted_beta_by_n": result["sep_weighted_beta_by_n"],
                "sep_weighted_beta_by_sxx": result["sep_weighted_beta_by_sxx"],
                "pooled_test_rmse": result["pooled_test_rmse"],
                "fe_test_rmse": result["fe_test_rmse"],
                "sep_test_rmse": result["sep_test_rmse"],
            }
        )

    sims = pd.DataFrame(sim_rows)

    summary = pd.DataFrame(
        {
            "method": [
                "Naive pooled",
                "Fixed effects",
                "Separate weighted by n",
                "Separate weighted by Sxx",
            ],
            "mean_beta": [
                sims["pooled_beta"].mean(),
                sims["fe_beta"].mean(),
                sims["sep_weighted_beta_by_n"].mean(),
                sims["sep_weighted_beta_by_sxx"].mean(),
            ],
            "sd_beta": [
                sims["pooled_beta"].std(),
                sims["fe_beta"].std(),
                sims["sep_weighted_beta_by_n"].std(),
                sims["sep_weighted_beta_by_sxx"].std(),
            ],
            "beta_rmse_vs_target": [
                rmse(np.full(n_sims, true_common_beta), sims["pooled_beta"]),
                rmse(np.full(n_sims, true_common_beta), sims["fe_beta"]),
                rmse(np.full(n_sims, true_common_beta), sims["sep_weighted_beta_by_n"]),
                rmse(np.full(n_sims, true_common_beta), sims["sep_weighted_beta_by_sxx"]),
            ],
        }
    )

    prediction_summary = pd.DataFrame(
        {
            "method": [
                "Naive pooled",
                "Fixed effects",
                "Separate regressions",
            ],
            "mean_test_rmse": [
                sims["pooled_test_rmse"].mean(),
                sims["fe_test_rmse"].mean(),
                sims["sep_test_rmse"].mean(),
            ],
            "sd_test_rmse": [
                sims["pooled_test_rmse"].std(),
                sims["fe_test_rmse"].std(),
                sims["sep_test_rmse"].std(),
            ],
        }
    )

    print()
    print("Monte Carlo Summary Over Repeated Training Samples")
    print("--------------------------------------------------")
    print(f"Scenario: {scenario}")
    print(f"Number of simulations: {n_sims}")
    print(f"Target beta for summary: {true_common_beta:.4f}")
    display(summary)

    print()
    print("Prediction Performance on New Test Data")
    print("---------------------------------------")
    display(prediction_summary)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].hist(
        sims["fe_beta"],
        bins=30,
        alpha=0.65,
        label="Fixed effects",
        color="tab:blue",
    )

    axes[0].hist(
        sims["sep_weighted_beta_by_n"],
        bins=30,
        alpha=0.65,
        label="Separate weighted by n",
        color="tab:orange",
    )

    axes[0].axvline(true_common_beta, color="black", linewidth=2, label="Target beta")
    axes[0].set_title("Sampling Distribution of Beta Estimates")
    axes[0].set_xlabel("Estimated beta")
    axes[0].set_ylabel("Count")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].boxplot(
        [
            sims["pooled_test_rmse"],
            sims["fe_test_rmse"],
            sims["sep_test_rmse"],
        ],
        labels=[
            "Pooled",
            "FE",
            "Separate",
        ],
    )

    axes[1].set_title("Test RMSE Across Simulations")
    axes[1].set_ylabel("Test RMSE")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    run_demo,

    scenario=Dropdown(
        options=["Common slope", "Group-specific slopes"],
        value="Common slope",
        description="Scenario",
    ),

    mean_x_A=FloatSlider(
        value=-3.0,
        min=-8.0,
        max=8.0,
        step=0.25,
        description="Mean X A",
        continuous_update=False,
    ),
    mean_x_B=FloatSlider(
        value=0.0,
        min=-8.0,
        max=8.0,
        step=0.25,
        description="Mean X B",
        continuous_update=False,
    ),
    mean_x_C=FloatSlider(
        value=3.0,
        min=-8.0,
        max=8.0,
        step=0.25,
        description="Mean X C",
        continuous_update=False,
    ),

    var_x_A=FloatSlider(
        value=1.0,
        min=0.05,
        max=20.0,
        step=0.05,
        description="Var X A",
        continuous_update=False,
    ),
    var_x_B=FloatSlider(
        value=1.0,
        min=0.05,
        max=20.0,
        step=0.05,
        description="Var X B",
        continuous_update=False,
    ),
    var_x_C=FloatSlider(
        value=1.0,
        min=0.05,
        max=20.0,
        step=0.05,
        description="Var X C",
        continuous_update=False,
    ),

    fixed_A=FloatSlider(
        value=-4.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="FE A",
        continuous_update=False,
    ),
    fixed_B=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="FE B",
        continuous_update=False,
    ),
    fixed_C=FloatSlider(
        value=4.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="FE C",
        continuous_update=False,
    ),

    beta_A=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="Beta A",
        continuous_update=False,
    ),
    beta_B=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="Beta B",
        continuous_update=False,
    ),
    beta_C=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="Beta C",
        continuous_update=False,
    ),

    n_A=IntSlider(
        value=10,
        min=4,
        max=100,
        step=1,
        description="N A",
        continuous_update=False,
    ),
    n_B=IntSlider(
        value=10,
        min=4,
        max=100,
        step=1,
        description="N B",
        continuous_update=False,
    ),
    n_C=IntSlider(
        value=10,
        min=4,
        max=100,
        step=1,
        description="N C",
        continuous_update=False,
    ),

    n_test_per_group=IntSlider(
        value=500,
        min=50,
        max=2000,
        step=50,
        description="N test/g",
        continuous_update=False,
    ),

    y_noise_sd=FloatSlider(
        value=2.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="Y noise",
        continuous_update=False,
    ),

    n_sims=IntSlider(
        value=500,
        min=50,
        max=3000,
        step=50,
        description="Simulations",
        continuous_update=False,
    ),

    show_lines=Checkbox(
        value=True,
        description="Show lines",
    ),

    seed=IntSlider(
        value=123,
        min=1,
        max=9999,
        step=1,
        description="Seed",
        continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='Scenario', options=('Common slope', 'Group-specific slopes'), valu…

# What fixed effects control for — and what they do not

## Case study: How does pollution (X) lead to hospital admissions (Y) in different cities at different times?

## Fixed effects control for stable differences across groups: city geography, hospital culture, store location, user baseline preference, school neighborhood, etc.

## They do not automatically control for time-varying confounders. If a city changes both its pollution policy and its hospital reporting system at the same time, city fixed effects alone may not solve the problem.

## Possible misconception to address:

## Misconception: “If I add fixed effects, the coefficient is causal.”

## Correction: Fixed effects remove one kind of confounding. You still need a credible identification argument.

# Why many fixed effects are common in real data

## In real causal analysis, groups are rarely just A, B, and C. They may be hundreds of stores, thousands of users, counties, hospitals, teachers, products, or firms.

## Fixed effects become especially useful when there are many groups with different baseline levels.

## Case studies:

## Retail: estimating whether a price change affects sales while controlling for store-level demand differences.

## Healthcare: estimating whether a treatment protocol affects readmission while controlling for hospital-level differences.

## Education: estimating whether attendance affects grades while controlling for student fixed effects.

## More fixed effects can reduce omitted-variable bias, but they also consume information. If each group has very few observations, separate slopes can become noisy.

# 100 Group Fixed Effects

In [5]:
import numpy as np
import pandas as pd

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
from IPython.display import display


def safe_positive_normal(mean, sd, size, rng, min_value=0.01):
    values = rng.normal(loc=mean, scale=sd, size=size)
    values = np.maximum(values, min_value)
    return values


def make_group_level_parameters(
    n_groups,
    x_mean_mean,
    x_mean_sd,
    x_variance_mean,
    x_variance_sd,
    fixed_effect_mean,
    fixed_effect_sd,
    x_fe_correlation,
    beta_common,
    slope_sd,
    scenario,
    rng,
):
    z_x_mean = rng.normal(0.0, 1.0, size=n_groups)
    z_independent = rng.normal(0.0, 1.0, size=n_groups)

    x_mean = x_mean_mean + x_mean_sd * z_x_mean

    fixed_effect_standardized = (
        x_fe_correlation * z_x_mean
        + np.sqrt(max(1.0 - x_fe_correlation ** 2, 0.0)) * z_independent
    )

    fixed_effect = fixed_effect_mean + fixed_effect_sd * fixed_effect_standardized

    x_variance = safe_positive_normal(
        mean=x_variance_mean,
        sd=x_variance_sd,
        size=n_groups,
        rng=rng,
        min_value=0.01,
    )

    if scenario == "Common slope":
        beta = np.full(n_groups, beta_common)
    else:
        beta = rng.normal(loc=beta_common, scale=slope_sd, size=n_groups)

    group_params = pd.DataFrame(
        {
            "group": [f"G{g:03d}" for g in range(n_groups)],
            "group_id": np.arange(n_groups),
            "x_mean": x_mean,
            "x_variance": x_variance,
            "fixed_effect": fixed_effect,
            "true_beta": beta,
        }
    )

    return group_params


def make_dataset_from_group_params(
    group_params,
    n_per_group,
    y_noise_sd,
    rng,
):
    rows = []

    for _, row in group_params.iterrows():
        x = rng.normal(
            loc=row["x_mean"],
            scale=np.sqrt(row["x_variance"]),
            size=n_per_group,
        )

        y = (
            row["fixed_effect"]
            + row["true_beta"] * x
            + rng.normal(0.0, y_noise_sd, size=n_per_group)
        )

        rows.append(
            pd.DataFrame(
                {
                    "group": row["group"],
                    "group_id": row["group_id"],
                    "x": x,
                    "y": y,
                }
            )
        )

    return pd.concat(rows, ignore_index=True)


def ols_slope_intercept(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    xbar = np.mean(x)
    ybar = np.mean(y)

    sxx = np.sum((x - xbar) ** 2)

    if sxx <= 1e-12:
        return np.nan, np.nan

    beta = np.sum((x - xbar) * (y - ybar)) / sxx
    intercept = ybar - beta * xbar

    return beta, intercept


def fit_pooled(df):
    beta, intercept = ols_slope_intercept(df["x"], df["y"])

    return {
        "intercept": intercept,
        "beta": beta,
    }


def fit_fixed_effects(df):
    work = df.copy()

    work["xbar_group"] = work.groupby("group")["x"].transform("mean")
    work["ybar_group"] = work.groupby("group")["y"].transform("mean")

    work["x_demeaned"] = work["x"] - work["xbar_group"]
    work["y_demeaned"] = work["y"] - work["ybar_group"]

    beta = (
        np.sum(work["x_demeaned"] * work["y_demeaned"])
        / np.sum(work["x_demeaned"] ** 2)
    )

    group_means = work.groupby("group")[["x", "y"]].mean()
    intercepts = group_means["y"] - beta * group_means["x"]

    return {
        "beta": beta,
        "intercepts": intercepts.to_dict(),
    }


def fit_separate(df):
    results = {}

    for group, sub in df.groupby("group"):
        beta, intercept = ols_slope_intercept(sub["x"], sub["y"])

        results[group] = {
            "beta": beta,
            "intercept": intercept,
            "n": len(sub),
        }

    return results


def predict_pooled(model, df):
    return model["intercept"] + model["beta"] * df["x"].to_numpy()


def predict_fixed_effects(model, df):
    intercepts = df["group"].map(model["intercepts"]).to_numpy()
    return intercepts + model["beta"] * df["x"].to_numpy()


def predict_separate(model, df):
    intercept_map = {g: model[g]["intercept"] for g in model}
    beta_map = {g: model[g]["beta"] for g in model}

    intercepts = df["group"].map(intercept_map).to_numpy()
    betas = df["group"].map(beta_map).to_numpy()

    return intercepts + betas * df["x"].to_numpy()


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


def mse(y_true, y_pred):
    return np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)


def run_one_simulation(
    n_groups,
    n_train_per_group,
    n_test_per_group,
    x_mean_mean,
    x_mean_sd,
    x_variance_mean,
    x_variance_sd,
    fixed_effect_mean,
    fixed_effect_sd,
    x_fe_correlation,
    beta_common,
    slope_sd,
    y_noise_sd,
    scenario,
    rng,
):
    group_params = make_group_level_parameters(
        n_groups=n_groups,
        x_mean_mean=x_mean_mean,
        x_mean_sd=x_mean_sd,
        x_variance_mean=x_variance_mean,
        x_variance_sd=x_variance_sd,
        fixed_effect_mean=fixed_effect_mean,
        fixed_effect_sd=fixed_effect_sd,
        x_fe_correlation=x_fe_correlation,
        beta_common=beta_common,
        slope_sd=slope_sd,
        scenario=scenario,
        rng=rng,
    )

    train = make_dataset_from_group_params(
        group_params=group_params,
        n_per_group=n_train_per_group,
        y_noise_sd=y_noise_sd,
        rng=rng,
    )

    test = make_dataset_from_group_params(
        group_params=group_params,
        n_per_group=n_test_per_group,
        y_noise_sd=y_noise_sd,
        rng=rng,
    )

    pooled = fit_pooled(train)
    fixed_effects = fit_fixed_effects(train)
    separate = fit_separate(train)

    pooled_pred = predict_pooled(pooled, test)
    fe_pred = predict_fixed_effects(fixed_effects, test)
    sep_pred = predict_separate(separate, test)

    true_average_beta = group_params["true_beta"].mean()

    separate_beta_simple_average = np.mean(
        [separate[g]["beta"] for g in separate]
    )

    sxx_by_group = {}

    for group, sub in train.groupby("group"):
        x = sub["x"].to_numpy()
        sxx_by_group[group] = np.sum((x - np.mean(x)) ** 2)

    total_sxx = sum(sxx_by_group.values())

    separate_beta_sxx_average = sum(
        separate[g]["beta"] * sxx_by_group[g] for g in separate
    ) / total_sxx

    return {
        "pooled_beta": pooled["beta"],
        "fixed_effects_beta": fixed_effects["beta"],
        "separate_beta_simple_average": separate_beta_simple_average,
        "separate_beta_sxx_average": separate_beta_sxx_average,
        "true_average_beta": true_average_beta,
        "pooled_rmse": rmse(test["y"], pooled_pred),
        "fixed_effects_rmse": rmse(test["y"], fe_pred),
        "separate_rmse": rmse(test["y"], sep_pred),
        "pooled_mse": mse(test["y"], pooled_pred),
        "fixed_effects_mse": mse(test["y"], fe_pred),
        "separate_mse": mse(test["y"], sep_pred),
        "actual_x_mean_mean": group_params["x_mean"].mean(),
        "actual_x_mean_sd": group_params["x_mean"].std(),
        "actual_x_variance_mean": group_params["x_variance"].mean(),
        "actual_x_variance_sd": group_params["x_variance"].std(),
        "actual_fixed_effect_mean": group_params["fixed_effect"].mean(),
        "actual_fixed_effect_sd": group_params["fixed_effect"].std(),
        "actual_xmean_fe_correlation": np.corrcoef(
            group_params["x_mean"],
            group_params["fixed_effect"],
        )[0, 1],
    }


def run_100_group_comparison(
    scenario="Common slope",
    n_groups=100,
    n_train_per_group=10,
    n_test_per_group=100,
    x_mean_mean=0.0,
    x_mean_sd=3.0,
    x_variance_mean=1.0,
    x_variance_sd=0.5,
    fixed_effect_mean=0.0,
    fixed_effect_sd=4.0,
    x_fe_correlation=0.8,
    beta_common=2.0,
    slope_sd=1.0,
    y_noise_sd=2.0,
    n_sims=500,
    seed=123,
):
    rng = np.random.default_rng(seed)

    rows = []

    for sim in range(n_sims):
        result = run_one_simulation(
            n_groups=n_groups,
            n_train_per_group=n_train_per_group,
            n_test_per_group=n_test_per_group,
            x_mean_mean=x_mean_mean,
            x_mean_sd=x_mean_sd,
            x_variance_mean=x_variance_mean,
            x_variance_sd=x_variance_sd,
            fixed_effect_mean=fixed_effect_mean,
            fixed_effect_sd=fixed_effect_sd,
            x_fe_correlation=x_fe_correlation,
            beta_common=beta_common,
            slope_sd=slope_sd,
            y_noise_sd=y_noise_sd,
            scenario=scenario,
            rng=rng,
        )

        rows.append(result)

    sims = pd.DataFrame(rows)

    noise_floor_rmse = y_noise_sd
    noise_floor_mse = y_noise_sd ** 2

    beta_summary = pd.DataFrame(
        {
            "method": [
                "Naive pooled",
                "Fixed effects",
                "Separate slopes, simple average",
                "Separate slopes, within-group Sxx average",
            ],
            "mean_beta": [
                sims["pooled_beta"].mean(),
                sims["fixed_effects_beta"].mean(),
                sims["separate_beta_simple_average"].mean(),
                sims["separate_beta_sxx_average"].mean(),
            ],
            "sd_beta": [
                sims["pooled_beta"].std(),
                sims["fixed_effects_beta"].std(),
                sims["separate_beta_simple_average"].std(),
                sims["separate_beta_sxx_average"].std(),
            ],
            "bias_vs_true_average_beta": [
                (sims["pooled_beta"] - sims["true_average_beta"]).mean(),
                (sims["fixed_effects_beta"] - sims["true_average_beta"]).mean(),
                (sims["separate_beta_simple_average"] - sims["true_average_beta"]).mean(),
                (sims["separate_beta_sxx_average"] - sims["true_average_beta"]).mean(),
            ],
            "rmse_vs_true_average_beta": [
                rmse(sims["true_average_beta"], sims["pooled_beta"]),
                rmse(sims["true_average_beta"], sims["fixed_effects_beta"]),
                rmse(sims["true_average_beta"], sims["separate_beta_simple_average"]),
                rmse(sims["true_average_beta"], sims["separate_beta_sxx_average"]),
            ],
        }
    )

    prediction_summary = pd.DataFrame(
        {
            "method": [
                "Naive pooled",
                "Fixed effects",
                "Separate regressions",
            ],
            "mean_test_rmse": [
                sims["pooled_rmse"].mean(),
                sims["fixed_effects_rmse"].mean(),
                sims["separate_rmse"].mean(),
            ],
            "sd_test_rmse": [
                sims["pooled_rmse"].std(),
                sims["fixed_effects_rmse"].std(),
                sims["separate_rmse"].std(),
            ],
            "mean_test_mse": [
                sims["pooled_mse"].mean(),
                sims["fixed_effects_mse"].mean(),
                sims["separate_mse"].mean(),
            ],
            "excess_mse_above_noise_floor": [
                sims["pooled_mse"].mean() - noise_floor_mse,
                sims["fixed_effects_mse"].mean() - noise_floor_mse,
                sims["separate_mse"].mean() - noise_floor_mse,
            ],
        }
    )

    fe_excess = prediction_summary.loc[
        prediction_summary["method"] == "Fixed effects",
        "excess_mse_above_noise_floor",
    ].iloc[0]

    sep_excess = prediction_summary.loc[
        prediction_summary["method"] == "Separate regressions",
        "excess_mse_above_noise_floor",
    ].iloc[0]

    pooled_excess = prediction_summary.loc[
        prediction_summary["method"] == "Naive pooled",
        "excess_mse_above_noise_floor",
    ].iloc[0]

    if sep_excess > 0:
        fe_vs_sep_excess_reduction = 100 * (sep_excess - fe_excess) / sep_excess
    else:
        fe_vs_sep_excess_reduction = np.nan

    if pooled_excess > 0:
        fe_vs_pooled_excess_reduction = 100 * (pooled_excess - fe_excess) / pooled_excess
    else:
        fe_vs_pooled_excess_reduction = np.nan

    parameter_summary = pd.DataFrame(
        {
            "quantity": [
                "Requested number of groups",
                "Training samples per group",
                "Test samples per group",
                "Simulations",
                "Noise floor RMSE",
                "Noise floor MSE",
                "Actual mean of group X means",
                "Actual SD of group X means",
                "Actual mean of group X variances",
                "Actual SD of group X variances",
                "Actual mean of fixed effects",
                "Actual SD of fixed effects",
                "Actual correlation: group X mean vs fixed effect",
                "FE excess-MSE reduction vs separate",
                "FE excess-MSE reduction vs pooled",
            ],
            "value": [
                n_groups,
                n_train_per_group,
                n_test_per_group,
                n_sims,
                noise_floor_rmse,
                noise_floor_mse,
                sims["actual_x_mean_mean"].mean(),
                sims["actual_x_mean_sd"].mean(),
                sims["actual_x_variance_mean"].mean(),
                sims["actual_x_variance_sd"].mean(),
                sims["actual_fixed_effect_mean"].mean(),
                sims["actual_fixed_effect_sd"].mean(),
                sims["actual_xmean_fe_correlation"].mean(),
                fe_vs_sep_excess_reduction,
                fe_vs_pooled_excess_reduction,
            ],
        }
    )

    print("100-Group Fixed Effects Comparison")
    print("----------------------------------")
    print(f"Scenario: {scenario}")
    print()
    print("Group-Level Parameter Summary")
    print("-----------------------------")
    display(parameter_summary)

    print()
    print("Beta Estimate Summary")
    print("---------------------")
    display(beta_summary)

    print()
    print("Prediction Summary")
    print("------------------")
    display(prediction_summary)

    print()
    print("Interpretation Aid")
    print("------------------")
    print(f"Noise floor RMSE is approximately {noise_floor_rmse:.4f}.")
    print(f"Noise floor MSE is approximately {noise_floor_mse:.4f}.")
    print()
    print(
        "The excess MSE above the noise floor estimates avoidable modeling error."
    )
    print(
        f"Fixed effects reduces excess MSE by {fe_vs_sep_excess_reduction:.2f}% "
        "relative to separate regressions."
    )
    print(
        f"Fixed effects reduces excess MSE by {fe_vs_pooled_excess_reduction:.2f}% "
        "relative to naive pooling."
    )


interact(
    run_100_group_comparison,

    scenario=Dropdown(
        options=["Common slope", "Group-specific slopes"],
        value="Common slope",
        description="Scenario",
    ),

    n_groups=IntSlider(
        value=100,
        min=10,
        max=300,
        step=10,
        description="Groups",
        continuous_update=False,
    ),

    n_train_per_group=IntSlider(
        value=10,
        min=3,
        max=100,
        step=1,
        description="Train/group",
        continuous_update=False,
    ),

    n_test_per_group=IntSlider(
        value=100,
        min=20,
        max=1000,
        step=20,
        description="Test/group",
        continuous_update=False,
    ),

    x_mean_mean=FloatSlider(
        value=0.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="Mean of X means",
        continuous_update=False,
    ),

    x_mean_sd=FloatSlider(
        value=3.0,
        min=0.0,
        max=10.0,
        step=0.25,
        description="SD of X means",
        continuous_update=False,
    ),

    x_variance_mean=FloatSlider(
        value=1.0,
        min=0.05,
        max=20.0,
        step=0.05,
        description="Mean X var",
        continuous_update=False,
    ),

    x_variance_sd=FloatSlider(
        value=0.5,
        min=0.0,
        max=10.0,
        step=0.05,
        description="SD X var",
        continuous_update=False,
    ),

    fixed_effect_mean=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="Mean FE",
        continuous_update=False,
    ),

    fixed_effect_sd=FloatSlider(
        value=4.0,
        min=0.0,
        max=15.0,
        step=0.25,
        description="SD FE",
        continuous_update=False,
    ),

    x_fe_correlation=FloatSlider(
        value=0.8,
        min=-0.99,
        max=0.99,
        step=0.01,
        description="Corr Xmean-FE",
        continuous_update=False,
    ),

    beta_common=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="Common beta",
        continuous_update=False,
    ),

    slope_sd=FloatSlider(
        value=1.0,
        min=0.0,
        max=5.0,
        step=0.25,
        description="SD slopes",
        continuous_update=False,
    ),

    y_noise_sd=FloatSlider(
        value=2.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description="Y noise",
        continuous_update=False,
    ),

    n_sims=IntSlider(
        value=500,
        min=50,
        max=3000,
        step=50,
        description="Simulations",
        continuous_update=False,
    ),

    seed=IntSlider(
        value=123,
        min=1,
        max=9999,
        step=1,
        description="Seed",
        continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='Scenario', options=('Common slope', 'Group-specific slopes'), valu…

# Overall R-squared vs. Within Groups R-squared

In [12]:
from IPython.display import display, HTML

display(HTML("""
<div style="font-size: 30px; line-height: 1.45; max-width: 1100px;">

<h1 style="font-size: 44px;">Fixed Effects with X: Overall R² and Within R²</h1>

<p>
Suppose we have three groups: <b>A</b>, <b>B</b>, and <b>C</b>.
For each observation, we observe:
</p>

<div style="background-color:#f3f3f3; padding: 15px;">
an outcome, <b>y</b>, and a predictor, <b>X</b>
</div>

<p>
We fit a fixed effects regression with one predictor:
</p>

<div style="background-color:#eaf4ff; padding: 20px;">
<b>predicted y = group fixed effect + β<sub>1</sub>X</b>
</div>

<p>
This model allows each group to have its own baseline, while using <b>X</b> to explain additional differences in <b>y</b>.
</p>

<hr>

<h2 style="font-size: 38px;">1. The outcome values</h2>

<p>
The observed outcomes are:
</p>

<table style="font-size: 30px; border-collapse: collapse;">
<tr>
  <th style="border: 1px solid black; padding: 8px;">Group A</th>
  <th style="border: 1px solid black; padding: 8px;">Group B</th>
  <th style="border: 1px solid black; padding: 8px;">Group C</th>
</tr>
<tr>
  <td style="border: 1px solid black; padding: 8px;">y<sub>A1</sub>, y<sub>A2</sub>, ..., y<sub>A nA</sub></td>
  <td style="border: 1px solid black; padding: 8px;">y<sub>B1</sub>, y<sub>B2</sub>, ..., y<sub>B nB</sub></td>
  <td style="border: 1px solid black; padding: 8px;">y<sub>C1</sub>, y<sub>C2</sub>, ..., y<sub>C nC</sub></td>
</tr>
</table>

<p>
The <b>grand mean</b> is the average of all outcome values:
</p>

<div style="background-color:#f3f3f3; padding: 15px;">
ȳ = average of all y values in groups A, B, and C together
</div>

<hr>

<h2 style="font-size: 38px;">2. The group means</h2>

<p>
Each group also has its own average outcome:
</p>

<div style="background-color:#f9f9f9; padding: 15px;">
ȳ<sub>A</sub> = average outcome in group A <br>
ȳ<sub>B</sub> = average outcome in group B <br>
ȳ<sub>C</sub> = average outcome in group C
</div>

<p>
These group means matter because fixed effects allow each group to have its own baseline.
</p>

<div style="background-color:#eaf4ff; padding: 15px;">
Group A has its own intercept. <br>
Group B has its own intercept. <br>
Group C has its own intercept.
</div>

<hr>

<h2 style="font-size: 38px;">3. The fitted model</h2>

<p>
The fitted model predicts each outcome using both the group fixed effect and X:
</p>

<div style="background-color:#eaf4ff; padding: 20px;">
ŷ = group fixed effect + β<sub>1</sub>X
</div>

<p>
Written by group:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
For group A: ŷ<sub>A1</sub>, ŷ<sub>A2</sub>, ..., ŷ<sub>A nA</sub> <br>
For group B: ŷ<sub>B1</sub>, ŷ<sub>B2</sub>, ..., ŷ<sub>B nB</sub> <br>
For group C: ŷ<sub>C1</sub>, ŷ<sub>C2</sub>, ..., ŷ<sub>C nC</sub>
</div>

<p>
Because X can vary within a group, two observations in the same group can have different predicted values.
</p>

<hr>

<h2 style="font-size: 38px;">4. Model leftover variation (residual)</h2>

<p>
After fitting the model, the leftover error for each observation is:
</p>

<div style="background-color:#f3f3f3; padding: 15px;">
actual outcome − predicted outcome
</div>

<p>
So the model leftover variation is:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
For group A: (y<sub>A1</sub> − ŷ<sub>A1</sub>)² + (y<sub>A2</sub> − ŷ<sub>A2</sub>)² + ... <br>
For group B: (y<sub>B1</sub> − ŷ<sub>B1</sub>)² + (y<sub>B2</sub> − ŷ<sub>B2</sub>)² + ... <br>
For group C: (y<sub>C1</sub> − ŷ<sub>C1</sub>)² + (y<sub>C2</sub> − ŷ<sub>C2</sub>)² + ...
</div>

<p>
This is the numerator used in both versions of R² below.
</p>

<div style="background-color:#d4edda; padding: 20px;">
<b>Same fitted model</b> → <b>same leftover variation</b>
</div>

<hr>

<h2 style="font-size: 38px;">5. Total variation</h2>

<p>
Total variation compares each outcome to the grand mean.
</p>

<div style="background-color:#fff3cd; padding: 15px;">
How far is each y value from the overall average y?
</div>

<p>
Written out by groups:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
For group A: (y<sub>A1</sub> − ȳ)² + (y<sub>A2</sub> − ȳ)² + ... <br>
For group B: (y<sub>B1</sub> − ȳ)² + (y<sub>B2</sub> − ȳ)² + ... <br>
For group C: (y<sub>C1</sub> − ȳ)² + (y<sub>C2</sub> − ȳ)² + ...
</div>

<p>
This is the denominator for <b>overall R²</b>.
</p>

<hr>

<h2 style="font-size: 38px;">6. Within-group variation</h2>

<p>
Within-group variation compares each outcome to its own group mean.
</p>

<div style="background-color:#fff3cd; padding: 15px;">
After group average differences are removed, how much variation is left inside the groups?
</div>

<p>
Written out by groups:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
For group A: (y<sub>A1</sub> − ȳ<sub>A</sub>)² + (y<sub>A2</sub> − ȳ<sub>A</sub>)² + ... <br>
For group B: (y<sub>B1</sub> − ȳ<sub>B</sub>)² + (y<sub>B2</sub> − ȳ<sub>B</sub>)² + ... <br>
For group C: (y<sub>C1</sub> − ȳ<sub>C</sub>)² + (y<sub>C2</sub> − ȳ<sub>C</sub>)² + ...
</div>

<p>
This is the denominator for <b>within R²</b>.
</p>

<hr>

<h2 style="font-size: 38px;">7. Overall R²</h2>

<p>
Overall R² compares the model's leftover variation to the total variation around the grand mean.
</p>

<div style="background-color:#eaf4ff; padding: 20px;">

<b>Overall R² = 1 −</b>

<br><br>

<div style="text-align:center;">
<table style="font-size: 30px; margin-left:auto; margin-right:auto;">
<tr>
<td style="border-bottom: 3px solid black; padding: 8px;">
model leftover variation
</td>
</tr>
<tr>
<td style="padding: 8px;">
total variation around the grand mean
</td>
</tr>
</table>
</div>

</div>

<p>
Using the terms above:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
<b>Overall R² = 1 −</b>

<br><br>

<div style="text-align:center;">
<table style="font-size: 30px; margin-left:auto; margin-right:auto;">
<tr>
<td style="border-bottom: 3px solid black; padding: 8px;">
(y<sub>A1</sub> − ŷ<sub>A1</sub>)² + (y<sub>A2</sub> − ŷ<sub>A2</sub>)² + ... <br>
+ (y<sub>B1</sub> − ŷ<sub>B1</sub>)² + (y<sub>B2</sub> − ŷ<sub>B2</sub>)² + ... <br>
+ (y<sub>C1</sub> − ŷ<sub>C1</sub>)² + (y<sub>C2</sub> − ŷ<sub>C2</sub>)² + ...
</td>
</tr>
<tr>
<td style="padding: 8px;">
(y<sub>A1</sub> − ȳ)² + (y<sub>A2</sub> − ȳ)² + ... <br>
+ (y<sub>B1</sub> − ȳ)² + (y<sub>B2</sub> − ȳ)² + ... <br>
+ (y<sub>C1</sub> − ȳ)² + (y<sub>C2</sub> − ȳ)² + ...
</td>
</tr>
</table>
</div>
</div>

<p>
This asks:
</p>

<div style="background-color:#fff3cd; padding: 20px;">
Compared with predicting the grand mean for everyone, how much variation is removed by using group fixed effects and X?
</div>

<hr>

<h2 style="font-size: 38px;">8. Within R²</h2>

<p>
Within R² compares the same model leftover variation to the within-group variation.
</p>

<div style="background-color:#eaf4ff; padding: 20px;">

<b>Within R² = 1 −</b>

<br><br>

<div style="text-align:center;">
<table style="font-size: 30px; margin-left:auto; margin-right:auto;">
<tr>
<td style="border-bottom: 3px solid black; padding: 8px;">
model leftover variation
</td>
</tr>
<tr>
<td style="padding: 8px;">
within-group variation
</td>
</tr>
</table>
</div>

</div>

<p>
Using the terms above:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
<b>Within R² = 1 −</b>

<br><br>

<div style="text-align:center;">
<table style="font-size: 30px; margin-left:auto; margin-right:auto;">
<tr>
<td style="border-bottom: 3px solid black; padding: 8px;">
(y<sub>A1</sub> − ŷ<sub>A1</sub>)² + (y<sub>A2</sub> − ŷ<sub>A2</sub>)² + ... <br>
+ (y<sub>B1</sub> − ŷ<sub>B1</sub>)² + (y<sub>B2</sub> − ŷ<sub>B2</sub>)² + ... <br>
+ (y<sub>C1</sub> − ŷ<sub>C1</sub>)² + (y<sub>C2</sub> − ŷ<sub>C2</sub>)² + ...
</td>
</tr>
<tr>
<td style="padding: 8px;">
(y<sub>A1</sub> − ȳ<sub>A</sub>)² + (y<sub>A2</sub> − ȳ<sub>A</sub>)² + ... <br>
+ (y<sub>B1</sub> − ȳ<sub>B</sub>)² + (y<sub>B2</sub> − ȳ<sub>B</sub>)² + ... <br>
+ (y<sub>C1</sub> − ȳ<sub>C</sub>)² + (y<sub>C2</sub> − ȳ<sub>C</sub>)² + ...
</td>
</tr>
</table>
</div>
</div>

<p>
This asks:
</p>

<div style="background-color:#fff3cd; padding: 20px;">
After removing group average differences, how much of the remaining within-group variation is removed by X?
</div>

<hr>

<h2 style="font-size: 38px;">9. Why the numerator is the same</h2>

<p>
Both formulas use the leftover variation from the same fitted model:
</p>

<div style="background-color:#eaf4ff; padding: 20px;">
predicted y = group fixed effect + β<sub>1</sub>X
</div>

<p>
So the numerator is the same:
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
(y − ŷ)² terms added over all observations
</div>

<p>
The difference is the comparison point in the denominator.
</p>

<div style="background-color:#fff3cd; padding: 20px;">
<ul>
<li><b>Overall R²</b> uses variation around the grand mean.</li>
<li><b>Within R²</b> uses variation around group means.</li>
</ul>
</div>

<hr>

<h2 style="font-size: 38px;">10. How to interpret the two values</h2>

<div style="background-color:#d4edda; padding: 20px;">
<ul>
<li><b>Overall R²</b> measures how much total variation is explained by the whole model: group fixed effects plus X.</li>
<li><b>Within R²</b> measures how much within-group variation is explained beyond the group means, mainly by X.</li>
</ul>
</div>

<p>
So both are valid, but they answer different questions.
</p>

<hr>

<h2 style="font-size: 38px;">11. Important special case</h2>

<p>
If the model had only group fixed effects and no X variable, then the model would predict each observation using only its group mean.
</p>

<div style="background-color:#f3f3f3; padding: 20px;">
For group A: predicted value = ȳ<sub>A</sub> <br>
For group B: predicted value = ȳ<sub>B</sub> <br>
For group C: predicted value = ȳ<sub>C</sub>
</div>

<p>
In that special case, the model leftover variation would equal the within-group variation.
Therefore:
</p>

<div style="background-color:#eaf4ff; padding: 20px;">
Within R² = 1 − within-group variation / within-group variation = 0
</div>

<p>
But with the model we are discussing here, the model also includes X.
If X explains variation inside the groups, then the model leftover variation can be smaller than the within-group variation.
</p>

<div style="background-color:#d4edda; padding: 20px;">
That is why within R² can be positive when the model includes X.
</div>

</div>
"""))

# Fixed Effects with Dummy Variables

from IPython.display import display, HTML

display(HTML("""
<div style="font-size: 28px; line-height: 1.45; max-width: 1100px;">

<h1 style="font-size: 42px;">Fixed Effects with Dummy Variables</h1>

<p>
A fixed-effects model can be written as a regression with dummy variables.
The dummy variables let each group have its own intercept.
</p>

<div style="background-color:#eaf4ff; padding: 18px; margin: 14px 0;">
<b>Main idea:</b> group fixed effects are group-specific intercepts.
</div>

<hr>

<h2 style="font-size: 36px;">Example: Three Groups</h2>

<p>
Suppose there are three groups: <b>A</b>, <b>B</b>, and <b>C</b>.
If the model includes an intercept, we use only two dummy variables:
</p>

<div style="background-color:#f3f3f3; padding: 18px; margin: 14px 0;">
<b>D<sub>B</sub> = 1</b> if the observation is in group B, and 0 otherwise.
<br>
<b>D<sub>C</sub> = 1</b> if the observation is in group C, and 0 otherwise.
</div>

<p>
Group <b>A</b> is left out and becomes the <b>reference group</b>.
</p>

<table style="font-size: 28px; border-collapse: collapse; margin-top: 18px; margin-bottom: 18px;">
  <tr style="background-color:#d9edf7;">
    <th style="border: 1px solid black; padding: 10px;">Group</th>
    <th style="border: 1px solid black; padding: 10px;">X</th>
    <th style="border: 1px solid black; padding: 10px;">D<sub>B</sub></th>
    <th style="border: 1px solid black; padding: 10px;">D<sub>C</sub></th>
    <th style="border: 1px solid black; padding: 10px;">Predicted Intercept</th>
  </tr>
  <tr>
    <td style="border: 1px solid black; padding: 10px;"><b>A</b></td>
    <td style="border: 1px solid black; padding: 10px;">2</td>
    <td style="border: 1px solid black; padding: 10px;">0</td>
    <td style="border: 1px solid black; padding: 10px;">0</td>
    <td style="border: 1px solid black; padding: 10px;">β<sub>0</sub></td>
  </tr>
  <tr>
    <td style="border: 1px solid black; padding: 10px;"><b>B</b></td>
    <td style="border: 1px solid black; padding: 10px;">2</td>
    <td style="border: 1px solid black; padding: 10px;">1</td>
    <td style="border: 1px solid black; padding: 10px;">0</td>
    <td style="border: 1px solid black; padding: 10px;">β<sub>0</sub> + γ<sub>B</sub></td>
  </tr>
  <tr>
    <td style="border: 1px solid black; padding: 10px;"><b>C</b></td>
    <td style="border: 1px solid black; padding: 10px;">2</td>
    <td style="border: 1px solid black; padding: 10px;">0</td>
    <td style="border: 1px solid black; padding: 10px;">1</td>
    <td style="border: 1px solid black; padding: 10px;">β<sub>0</sub> + γ<sub>C</sub></td>
  </tr>
</table>

<hr>

<h2 style="font-size: 36px;">The Regression Equation</h2>

<p>
The dummy-variable regression is:
</p>

<div style="background-color:#eaf4ff; padding: 20px; margin: 14px 0;">
<b>
y<sub>i</sub> =
β<sub>0</sub>
+
β<sub>1</sub>X<sub>i</sub>
+
γ<sub>B</sub>D<sub>B,i</sub>
+
γ<sub>C</sub>D<sub>C,i</sub>
+
ε<sub>i</sub>
</b>
</div>

<p>
The intercept for group A is <b>β<sub>0</sub></b>.
The coefficient <b>γ<sub>B</sub></b> tells us how much group B's intercept differs from group A's intercept.
The coefficient <b>γ<sub>C</sub></b> tells us how much group C's intercept differs from group A's intercept.
</p>

<div style="background-color:#d4edda; padding: 18px; margin: 14px 0;">
<ul>
  <li><b>Group A intercept:</b> β<sub>0</sub></li>
  <li><b>Group B intercept:</b> β<sub>0</sub> + γ<sub>B</sub></li>
  <li><b>Group C intercept:</b> β<sub>0</sub> + γ<sub>C</sub></li>
</ul>
</div>

<hr>

<h2 style="font-size: 36px;">Diagram: One Group Is the Reference Group</h2>

<div style="display: flex; align-items: center; gap: 22px; margin-top: 20px; margin-bottom: 20px;">

  <div style="background-color:#f8d7da; border: 2px solid #721c24; border-radius: 10px; padding: 20px; width: 220px; text-align: center;">
    <div style="font-size: 32px;"><b>Group A</b></div>
    <div style="font-size: 24px; margin-top: 10px;">Reference group</div>
    <div style="font-size: 24px; margin-top: 10px;">No dummy variable</div>
    <div style="font-size: 28px; margin-top: 10px;"><b>β<sub>0</sub></b></div>
  </div>

  <div style="font-size: 36px;">+</div>

  <div style="background-color:#fff3cd; border: 2px solid #856404; border-radius: 10px; padding: 20px; width: 220px; text-align: center;">
    <div style="font-size: 32px;"><b>Group B</b></div>
    <div style="font-size: 24px; margin-top: 10px;">Dummy variable</div>
    <div style="font-size: 28px; margin-top: 10px;">D<sub>B</sub></div>
    <div style="font-size: 28px; margin-top: 10px;"><b>β<sub>0</sub> + γ<sub>B</sub></b></div>
  </div>

  <div style="font-size: 36px;">+</div>

  <div style="background-color:#d4edda; border: 2px solid #155724; border-radius: 10px; padding: 20px; width: 220px; text-align: center;">
    <div style="font-size: 32px;"><b>Group C</b></div>
    <div style="font-size: 24px; margin-top: 10px;">Dummy variable</div>
    <div style="font-size: 28px; margin-top: 10px;">D<sub>C</sub></div>
    <div style="font-size: 28px; margin-top: 10px;"><b>β<sub>0</sub> + γ<sub>C</sub></b></div>
  </div>

</div>

<div style="background-color:#fff3cd; padding: 18px; margin: 14px 0;">
<b>With 3 groups, use 2 dummy variables if the model includes an intercept.</b>
<br>
One group becomes the reference group.
</div>

<hr>

<h2 style="font-size: 36px;">Why Do We Leave One Group Out?</h2>

<p>
If we include an intercept and all three group dummy variables, the columns are perfectly collinear.
This problem is called the <b>dummy variable trap</b>.
</p>

<p>
For every observation, exactly one of the three group dummies equals 1:
</p>

<div style="background-color:#f3f3f3; padding: 18px; margin: 14px 0;">
D<sub>A</sub> + D<sub>B</sub> + D<sub>C</sub> = 1
</div>

<p>
But the intercept column is also a column of 1s.
So the intercept column contains the same information as the sum of the three dummy columns.
</p>

<div style="background-color:#f8d7da; padding: 18px; margin: 14px 0;">
<b>Dummy variable trap:</b>
<br>
Intercept = D<sub>A</sub> + D<sub>B</sub> + D<sub>C</sub>
<br><br>
That means the regression matrix has redundant columns, so the model cannot uniquely estimate all coefficients.
</div>

<div style="background-color:#d4edda; padding: 18px; margin: 14px 0;">
<b>Solution:</b>
<br>
If the model has an intercept, omit one dummy variable.
The omitted group becomes the reference group.
</div>

</div>
"""))

# Prerequisite insight: one fixed effect removes one kind of baseline difference

## A city fixed effect gives each city its own baseline.

## A person fixed effect gives each person their own baseline.

## Two-way fixed effects allow two separate baseline adjustments at the same time.

## In many causal settings, we want to control for unit differences and time differences. For example, store fixed effects control for stable differences across stores, while month fixed effects control for seasonality shared by all stores.

# Common causal version:

## outcome_it = beta_1 * treatment_it + unit fixed effect_i + time fixed effect_t + error_it

# Possible misconception:

## Misconception: two-way fixed effects means every store-time combination gets its own intercept.

## Correction: in an additive two-way model, the cell intercept is built from a store effect plus a time effect. It is not a completely unrestricted intercept for every cell.

# Two Way Fixed Effects

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from IPython.display import display


def make_two_way_data(
    n_per_cell,
    mean_x,
    var_x,
    intercept,
    beta_alice,
    beta_bob,
    beta_carol,
    beta_chicago,
    beta_denver,
    beta_boston,
    beta_1,
    y_noise_sd,
    seed,
):
    rng = np.random.default_rng(seed)

    people = ["Alice", "Bob", "Carol"]
    cities = ["Chicago", "Denver", "Boston"]

    person_effects = {
        "Alice": beta_alice,
        "Bob": beta_bob,
        "Carol": beta_carol,
    }

    city_effects = {
        "Chicago": beta_chicago,
        "Denver": beta_denver,
        "Boston": beta_boston,
    }

    rows = []

    for person in people:
        for city in cities:
            group_name = f"{person}_{city}"

            x = rng.normal(
                loc=mean_x,
                scale=np.sqrt(var_x),
                size=n_per_cell,
            )

            y_without_noise = (
                intercept
                + person_effects[person]
                + city_effects[city]
                + beta_1 * x
            )

            y = y_without_noise + rng.normal(
                loc=0.0,
                scale=y_noise_sd,
                size=n_per_cell,
            )

            for i in range(n_per_cell):
                rows.append(
                    {
                        "person": person,
                        "city": city,
                        "group": group_name,
                        "x": x[i],
                        "y": y[i],
                        "true_intercept": (
                            intercept
                            + person_effects[person]
                            + city_effects[city]
                        ),
                    }
                )

    return pd.DataFrame(rows)


def run_two_way_fixed_effects_demo(
    beta_alice=-4.0,
    beta_bob=0.0,
    beta_carol=4.0,
    beta_chicago=-3.0,
    beta_denver=0.0,
    beta_boston=3.0,
    beta_1=2.0,
    intercept=10.0,
    mean_x=0.0,
    var_x=4.0,
    n_per_cell=12,
    y_noise_sd=1.5,
    show_true_lines=True,
    seed=123,
):
    df = make_two_way_data(
        n_per_cell=n_per_cell,
        mean_x=mean_x,
        var_x=var_x,
        intercept=intercept,
        beta_alice=beta_alice,
        beta_bob=beta_bob,
        beta_carol=beta_carol,
        beta_chicago=beta_chicago,
        beta_denver=beta_denver,
        beta_boston=beta_boston,
        beta_1=beta_1,
        y_noise_sd=y_noise_sd,
        seed=seed,
    )

    people = ["Alice", "Bob", "Carol"]
    cities = ["Chicago", "Denver", "Boston"]

    colors = {
        "Alice": "tab:blue",
        "Bob": "tab:orange",
        "Carol": "tab:green",
    }

    markers = {
        "Chicago": "o",
        "Denver": "s",
        "Boston": "^",
    }

    x_min = df["x"].min()
    x_max = df["x"].max()
    x_margin = 0.15 * max(x_max - x_min, 1.0)
    x_grid = np.linspace(x_min - x_margin, x_max + x_margin, 200)

    fig, ax = plt.subplots(figsize=(12, 8))

    for person in people:
        for city in cities:
            sub = df[(df["person"] == person) & (df["city"] == city)]
            group_name = f"{person}_{city}"

            ax.scatter(
                sub["x"],
                sub["y"],
                color=colors[person],
                marker=markers[city],
                s=70,
                alpha=0.8,
                edgecolor="black",
                linewidth=0.4,
                label=group_name,
            )

            if show_true_lines:
                cell_intercept = sub["true_intercept"].iloc[0]

                ax.plot(
                    x_grid,
                    cell_intercept + beta_1 * x_grid,
                    color=colors[person],
                    linestyle="-",
                    linewidth=2,
                    alpha=0.7,
                )

    ax.set_title(
        "Two-Way Fixed Effects: Person Effects + City Effects",
        fontsize=18,
    )
    ax.set_xlabel("X", fontsize=14)
    ax.set_ylabel("Y", fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(
        title="Person_City group",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
    )

    plt.tight_layout()
    plt.show()

    display_df = pd.DataFrame(
        {
            "cell": [
                "Alice_Chicago",
                "Alice_Denver",
                "Alice_Boston",
                "Bob_Chicago",
                "Bob_Denver",
                "Bob_Boston",
                "Carol_Chicago",
                "Carol_Denver",
                "Carol_Boston",
            ],
            "model": [
                "Y_Alice_Chicago = intercept + beta_Alice + beta_Chicago + beta_1 X_Alice_Chicago + epsilon_Alice_Chicago",
                "Y_Alice_Denver = intercept + beta_Alice + beta_Denver + beta_1 X_Alice_Denver + epsilon_Alice_Denver",
                "Y_Alice_Boston = intercept + beta_Alice + beta_Boston + beta_1 X_Alice_Boston + epsilon_Alice_Boston",
                "Y_Bob_Chicago = intercept + beta_Bob + beta_Chicago + beta_1 X_Bob_Chicago + epsilon_Bob_Chicago",
                "Y_Bob_Denver = intercept + beta_Bob + beta_Denver + beta_1 X_Bob_Denver + epsilon_Bob_Denver",
                "Y_Bob_Boston = intercept + beta_Bob + beta_Boston + beta_1 X_Bob_Boston + epsilon_Bob_Boston",
                "Y_Carol_Chicago = intercept + beta_Carol + beta_Chicago + beta_1 X_Carol_Chicago + epsilon_Carol_Chicago",
                "Y_Carol_Denver = intercept + beta_Carol + beta_Denver + beta_1 X_Carol_Denver + epsilon_Carol_Denver",
                "Y_Carol_Boston = intercept + beta_Carol + beta_Boston + beta_1 X_Carol_Boston + epsilon_Carol_Boston",
            ],
            "numeric_intercept": [
                intercept + beta_alice + beta_chicago,
                intercept + beta_alice + beta_denver,
                intercept + beta_alice + beta_boston,
                intercept + beta_bob + beta_chicago,
                intercept + beta_bob + beta_denver,
                intercept + beta_bob + beta_boston,
                intercept + beta_carol + beta_chicago,
                intercept + beta_carol + beta_denver,
                intercept + beta_carol + beta_boston,
            ],
        }
    )

    print("Two-way fixed-effects model:")
    print()
    print("Y_person_city = intercept + beta_person + beta_city + beta_1 X_person_city + epsilon_person_city")
    print()
    display(display_df)


interact(
    run_two_way_fixed_effects_demo,

    beta_alice=FloatSlider(
        value=-4.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Alice",
        continuous_update=False,
    ),

    beta_bob=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Bob",
        continuous_update=False,
    ),

    beta_carol=FloatSlider(
        value=4.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Carol",
        continuous_update=False,
    ),

    beta_chicago=FloatSlider(
        value=-3.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Chicago",
        continuous_update=False,
    ),

    beta_denver=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Denver",
        continuous_update=False,
    ),

    beta_boston=FloatSlider(
        value=3.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="β Boston",
        continuous_update=False,
    ),

    beta_1=FloatSlider(
        value=2.0,
        min=-5.0,
        max=5.0,
        step=0.25,
        description="β1 X",
        continuous_update=False,
    ),

    intercept=FloatSlider(
        value=10.0,
        min=-20.0,
        max=20.0,
        step=0.5,
        description="Intercept",
        continuous_update=False,
    ),

    mean_x=FloatSlider(
        value=0.0,
        min=-10.0,
        max=10.0,
        step=0.25,
        description="Mean X",
        continuous_update=False,
    ),

    var_x=FloatSlider(
        value=4.0,
        min=0.1,
        max=20.0,
        step=0.1,
        description="Var X",
        continuous_update=False,
    ),

    n_per_cell=IntSlider(
        value=12,
        min=3,
        max=100,
        step=1,
        description="N/cell",
        continuous_update=False,
    ),

    y_noise_sd=FloatSlider(
        value=1.5,
        min=0.0,
        max=10.0,
        step=0.1,
        description="Y noise",
        continuous_update=False,
    ),

    show_true_lines=Checkbox(
        value=True,
        description="Show lines",
    ),

    seed=IntSlider(
        value=123,
        min=1,
        max=9999,
        step=1,
        description="Seed",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=-4.0, continuous_update=False, description='β Alice', max=10.0, min=-1…

# Moving on to general concerns about regression

## Regression can produce a coefficient even when the uncertainty calculation is wrong. Heteroskedasticity and autocorrelation are two ways the usual standard errors can become misleading.

## Case study: advertising spend and sales. Small stores may have stable sales, while large stores have much more volatile sales. If high-leverage large stores also have high variance, conventional standard errors may understate uncertainty.

# Heteroskedasticity

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from IPython.display import display


def fit_ols(x, y):
    X = sm.add_constant(x)
    return sm.OLS(y, X).fit()


def make_sigma(x, sigma0=0.5, hetero_strength=3.0, pattern="increases_with_x"):
    if pattern == "homoskedastic":
        sigma = np.full_like(x, sigma0, dtype=float)

    elif pattern == "increases_with_x":
        sigma = sigma0 * (1.0 + hetero_strength * x / 10.0)

    elif pattern == "decreases_with_x":
        sigma = sigma0 * (1.0 + hetero_strength * (10.0 - x) / 10.0)

    elif pattern == "larger_at_extremes":
        sigma = sigma0 * (1.0 + hetero_strength * np.abs(x - 5.0) / 5.0)

    elif pattern == "larger_in_middle":
        sigma = sigma0 * (1.0 + hetero_strength * (1.0 - np.abs(x - 5.0) / 5.0))

    else:
        raise ValueError("Unknown heteroskedasticity pattern.")

    return sigma


def monte_carlo_slope_se(
    n=250,
    beta0=2.0,
    beta1=1.5,
    sigma0=0.5,
    hetero_strength=3.0,
    pattern="increases_with_x",
    R=1000,
    seed=999
):
    rng = np.random.default_rng(seed)

    slopes = np.empty(R)
    conventional_ses = np.empty(R)
    robust_hc3_ses = np.empty(R)
    robust_hc0_ses = np.empty(R)

    for r in range(R):
        x = rng.uniform(0, 10, size=n)
        sigma = make_sigma(
            x,
            sigma0=sigma0,
            hetero_strength=hetero_strength,
            pattern=pattern
        )

        y_true = beta0 + beta1 * x
        y = y_true + rng.normal(0, sigma)

        model = fit_ols(x, y)

        slopes[r] = model.params[1]
        conventional_ses[r] = model.bse[1]
        robust_hc3_ses[r] = model.get_robustcov_results(cov_type="HC3").bse[1]
        robust_hc0_ses[r] = model.get_robustcov_results(cov_type="HC0").bse[1]

    empirical_slope_se = slopes.std(ddof=1)

    return slopes, conventional_ses, robust_hc3_ses, robust_hc0_ses, empirical_slope_se


def draw_heteroskedasticity_demo(
    n=250,
    beta0=2.0,
    beta1=1.5,
    sigma0=0.5,
    hetero_strength=3.0,
    pattern="larger_at_extremes",
    R=1000,
    seed=12
):
    rng = np.random.default_rng(seed)

    x = rng.uniform(0, 10, size=n)
    x = np.sort(x)

    sigma = make_sigma(
        x,
        sigma0=sigma0,
        hetero_strength=hetero_strength,
        pattern=pattern
    )

    y_true = beta0 + beta1 * x
    y = y_true + rng.normal(0, sigma)

    model = fit_ols(x, y)

    conventional_se = model.bse[1]
    robust_hc0_se = model.get_robustcov_results(cov_type="HC0").bse[1]
    robust_hc3_se = model.get_robustcov_results(cov_type="HC3").bse[1]

    slopes, conventional_ses, robust_hc3_ses, robust_hc0_ses, empirical_slope_se = monte_carlo_slope_se(
        n=n,
        beta0=beta0,
        beta1=beta1,
        sigma0=sigma0,
        hetero_strength=hetero_strength,
        pattern=pattern,
        R=R,
        seed=seed + 1000
    )

    x_grid = np.linspace(0, 10, 400)
    y_grid = beta0 + beta1 * x_grid
    sigma_grid = make_sigma(
        x_grid,
        sigma0=sigma0,
        hetero_strength=hetero_strength,
        pattern=pattern
    )

    fitted_grid = model.params[0] + model.params[1] * x_grid

    fig, axes = plt.subplots(2, 2, figsize=(17, 12))

    ax = axes[0, 0]
    ax.scatter(x, y, alpha=0.65, s=45, color="darkorange", label="data")
    ax.plot(x_grid, y_grid, color="black", linewidth=3, label="true line")
    ax.plot(x_grid, fitted_grid, color="darkred", linewidth=3, label="OLS fitted line")
    ax.fill_between(
        x_grid,
        y_grid - 2 * sigma_grid,
        y_grid + 2 * sigma_grid,
        color="orange",
        alpha=0.18,
        label="about ±2σ(x)"
    )
    ax.set_title("One simulated dataset", fontsize=17)
    ax.set_xlabel("x", fontsize=14)
    ax.set_ylabel("y", fontsize=14)
    ax.legend(fontsize=10)

    ax = axes[0, 1]
    ax.plot(x_grid, sigma_grid, color="purple", linewidth=4)
    ax.scatter(x, sigma, color="purple", alpha=0.35, s=25)
    ax.set_title("Error standard deviation pattern", fontsize=17)
    ax.set_xlabel("x", fontsize=14)
    ax.set_ylabel("σ(x)", fontsize=14)
    ax.grid(alpha=0.25)

    ax = axes[1, 0]
    ax.hist(slopes, bins=45, color="steelblue", alpha=0.75, edgecolor="white")
    ax.axvline(beta1, color="gold", linewidth=3, linestyle="--", label="true slope")
    ax.axvline(slopes.mean(), color="black", linewidth=3, label="mean estimated slope")
    ax.axvline(model.params[1], color="darkred", linewidth=3, linestyle=":", label="single-data slope")
    ax.set_title(
        f"Monte Carlo slope estimates\nempirical SD = {empirical_slope_se:.4f}",
        fontsize=17
    )
    ax.set_xlabel("estimated slope", fontsize=14)
    ax.set_ylabel("count", fontsize=14)
    ax.legend(fontsize=10)

    ax = axes[1, 1]
    labels = [
        "Empirical\nslope SD",
        "Mean conventional\nOLS SE",
        "Mean robust\nHC0 SE",
        "Mean robust\nHC3 SE"
    ]
    values = [
        empirical_slope_se,
        conventional_ses.mean(),
        robust_hc0_ses.mean(),
        robust_hc3_ses.mean()
    ]
    colors = ["black", "gray", "skyblue", "royalblue"]

    ax.bar(labels, values, color=colors, alpha=0.8)
    ax.set_title("Repeated-sampling SE comparison", fontsize=17)
    ax.set_ylabel("standard error of slope", fontsize=14)
    ax.grid(axis="y", alpha=0.25)

    for i, value in enumerate(values):
        ax.text(i, value, f"{value:.4f}", ha="center", va="bottom", fontsize=12)

    plt.tight_layout()
    plt.show()

    print("Design:")
    print(f"  n = {n}")
    print(f"  x distribution = Uniform(0, 10)")
    print(f"  pattern = {pattern}")
    print(f"  Monte Carlo repetitions R = {R}")
    print()
    print("True model:")
    print(f"  y = {beta0:.3f} + {beta1:.3f}x + error")
    print(f"  E[error | x] = 0")
    print()
    print("Error standard deviation:")
    print(f"  sigma0 = {sigma0:.3f}")
    print(f"  hetero_strength = {hetero_strength:.3f}")
    print(f"  min sigma = {sigma.min():.3f}")
    print(f"  max sigma = {sigma.max():.3f}")
    print(f"  sqrt(mean variance) = {np.sqrt(np.mean(sigma ** 2)):.3f}")
    print()
    print("Single fitted dataset:")
    print(f"  estimated slope                    = {model.params[1]:.4f}")
    print(f"  conventional OLS slope SE          = {conventional_se:.4f}")
    print(f"  robust HC0 slope SE                = {robust_hc0_se:.4f}")
    print(f"  robust HC3 slope SE                = {robust_hc3_se:.4f}")
    print(f"  HC3 / conventional ratio           = {robust_hc3_se / conventional_se:.3f}")
    print()
    print("Monte Carlo repeated-sampling check:")
    print(f"  empirical SD of slope estimates    = {empirical_slope_se:.4f}")
    print(f"  mean conventional OLS slope SE      = {conventional_ses.mean():.4f}")
    print(f"  mean robust HC0 slope SE            = {robust_hc0_ses.mean():.4f}")
    print(f"  mean robust HC3 slope SE            = {robust_hc3_ses.mean():.4f}")
    print()
    print("Ratios:")
    print(f"  empirical / conventional            = {empirical_slope_se / conventional_ses.mean():.3f}")
    print(f"  empirical / robust HC0              = {empirical_slope_se / robust_hc0_ses.mean():.3f}")
    print(f"  empirical / robust HC3              = {empirical_slope_se / robust_hc3_ses.mean():.3f}")


interact(
    draw_heteroskedasticity_demo,
    n=IntSlider(value=250, min=60, max=1000, step=20, description="n"),
    beta0=FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.5, description="beta0"),
    beta1=FloatSlider(value=1.5, min=-3.0, max=3.0, step=0.25, description="beta1"),
    sigma0=FloatSlider(value=0.5, min=0.05, max=3.0, step=0.05, description="sigma0"),
    hetero_strength=FloatSlider(value=3.0, min=0.0, max=10.0, step=0.25, description="strength"),
    pattern=Dropdown(
        options=[
            ("Homoskedastic: constant variance", "homoskedastic"),
            ("Generic: variance increases with x", "increases_with_x"),
            ("Generic: variance decreases with x", "decreases_with_x"),
            ("Clearer: variance larger at extremes", "larger_at_extremes"),
            ("Opposite: variance larger in middle", "larger_in_middle")
        ],
        value="larger_at_extremes",
        description="pattern"
    ),
    R=IntSlider(value=1000, min=200, max=5000, step=200, description="MC R"),
    seed=IntSlider(value=12, min=1, max=999, step=1, description="seed")
)

interactive(children=(IntSlider(value=250, description='n', max=1000, min=60, step=20), FloatSlider(value=2.0,…

<function __main__.draw_heteroskedasticity_demo(n=250, beta0=2.0, beta1=1.5, sigma0=0.5, hetero_strength=3.0, pattern='larger_at_extremes', R=1000, seed=12)>

## Case study: daily app engagement. If today’s unexplained engagement is high, tomorrow’s may also be high because of a campaign, news event, or weekly habit. Treating every day as independent exaggerates the amount of information in the data.

# Regression with Autocorrelated Data

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from ipywidgets import interact, IntSlider, FloatSlider


def fit_ols(x, y):
    X = sm.add_constant(x)
    return sm.OLS(y, X).fit()


def make_ar1_errors(n, rho, sigma, rng):
    errors = np.empty(n)
    errors[0] = rng.normal(0, sigma)

    innovation_sigma = sigma * np.sqrt(1 - rho**2)

    for t in range(1, n):
        errors[t] = rho * errors[t - 1] + rng.normal(0, innovation_sigma)

    return errors


def ordinary_pairs_bootstrap_slope_se(x, y, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    n = len(x)
    slopes = np.empty(B)

    for b in range(B):
        idx = rng.integers(0, n, size=n)
        xb = x[idx]
        yb = y[idx]
        slopes[b] = fit_ols(xb, yb).params[1]

    return slopes.std(ddof=1), slopes


def moving_block_bootstrap_slope_se(x, y, block_length=10, B=1000, seed=123):
    rng = np.random.default_rng(seed)
    n = len(x)

    if block_length > n:
        block_length = n

    starts = np.arange(0, n - block_length + 1)
    n_blocks = int(np.ceil(n / block_length))

    slopes = np.empty(B)

    for b in range(B):
        chosen_starts = rng.choice(starts, size=n_blocks, replace=True)
        idx = []

        for start in chosen_starts:
            idx.extend(range(start, start + block_length))

        idx = np.array(idx[:n])

        xb = x[idx]
        yb = y[idx]
        slopes[b] = fit_ols(xb, yb).params[1]

    return slopes.std(ddof=1), slopes


def draw_demo(
    n=250,
    beta0=2.0,
    beta1=0.06,
    rho=0.80,
    sigma=2.0,
    x_noise=0.30,
    B=1000,
    block_length=15,
    seed=7
):
    rng = np.random.default_rng(seed)

    time = np.arange(n)

    x = time + rng.normal(0, x_noise * n / 10, size=n)
    x = (x - x.mean()) / x.std()

    errors = make_ar1_errors(n=n, rho=rho, sigma=sigma, rng=rng)

    y = beta0 + beta1 * x + errors

    model = fit_ols(x, y)

    conventional_se = model.bse[1]
    robust_hc3_se = model.get_robustcov_results(cov_type="HC3").bse[1]

    ordinary_boot_se, ordinary_boot_slopes = ordinary_pairs_bootstrap_slope_se(
        x,
        y,
        B=B,
        seed=seed + 1000
    )

    block_boot_se, block_boot_slopes = moving_block_bootstrap_slope_se(
        x,
        y,
        block_length=block_length,
        B=B,
        seed=seed + 2000
    )

    x_grid = np.linspace(x.min(), x.max(), 300)
    y_fit_grid = model.params[0] + model.params[1] * x_grid
    y_true_grid = beta0 + beta1 * x_grid

    slope_grid_conventional = np.linspace(
        model.params[1] - 2 * conventional_se,
        model.params[1] + 2 * conventional_se,
        9
    )

    slope_grid_block = np.linspace(
        model.params[1] - 2 * block_boot_se,
        model.params[1] + 2 * block_boot_se,
        9
    )

    x_pivot = x.mean()
    y_pivot = model.params[0] + model.params[1] * x_pivot

    def line_through_pivot(slope):
        return y_pivot + slope * (x_grid - x_pivot)

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    ax = axes[0, 0]
    sc = ax.scatter(x, y, c=time, cmap="viridis", s=40, alpha=0.75, label="one observed sample")
    ax.plot(x_grid, y_fit_grid, color="black", linewidth=4, label="OLS fitted line")
    ax.plot(x_grid, y_true_grid, color="gold", linewidth=3, linestyle="--", label="true line")

    for slope in slope_grid_conventional:
        ax.plot(
            x_grid,
            line_through_pivot(slope),
            color="red",
            alpha=0.35,
            linewidth=2
        )

    ax.set_title("One sample: OLS lines using independence-based SE", fontsize=17)
    ax.set_xlabel("X", fontsize=14)
    ax.set_ylabel("y", fontsize=14)
    ax.legend(fontsize=10)
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label("time order", fontsize=12)

    ax = axes[0, 1]
    ax.plot(time, errors, color="purple", linewidth=2)
    ax.axhline(0, color="black", linewidth=2)
    ax.set_title(f"Temporally correlated errors, rho = {rho:.2f}", fontsize=17)
    ax.set_xlabel("time", fontsize=14)
    ax.set_ylabel("error", fontsize=14)

    ax = axes[1, 0]
    ax.hist(
        ordinary_boot_slopes,
        bins=40,
        color="gray",
        alpha=0.75,
        edgecolor="white",
        label=f"ordinary pairs bootstrap SE = {ordinary_boot_se:.4f}"
    )
    ax.axvline(model.params[1], color="black", linewidth=3, label="original OLS slope")
    ax.axvline(beta1, color="gold", linewidth=3, linestyle="--", label="true slope")
    ax.axvline(model.params[1] - 2 * conventional_se, color="red", linewidth=2, linestyle=":")
    ax.axvline(model.params[1] + 2 * conventional_se, color="red", linewidth=2, linestyle=":")
    ax.set_title("Ordinary bootstrap: resamples points as if independent", fontsize=17)
    ax.set_xlabel("bootstrapped slope estimates", fontsize=14)
    ax.set_ylabel("count", fontsize=14)
    ax.legend(fontsize=10)

    ax = axes[1, 1]
    ax.hist(
        block_boot_slopes,
        bins=40,
        color="teal",
        alpha=0.75,
        edgecolor="white",
        label=f"block bootstrap SE = {block_boot_se:.4f}"
    )
    ax.axvline(model.params[1], color="black", linewidth=3, label="original OLS slope")
    ax.axvline(beta1, color="gold", linewidth=3, linestyle="--", label="true slope")
    ax.axvline(model.params[1] - 2 * conventional_se, color="red", linewidth=2, linestyle=":", label="±2 conventional SE")
    ax.axvline(model.params[1] + 2 * conventional_se, color="red", linewidth=2, linestyle=":")
    ax.set_title("Block bootstrap: resamples nearby time points together", fontsize=17)
    ax.set_xlabel("bootstrapped slope estimates", fontsize=14)
    ax.set_ylabel("count", fontsize=14)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Model:")
    print(f"  y_t = {beta0:.3f} + {beta1:.3f} X_t + error_t")
    print()
    print("Temporal correlation:")
    print(f"  error_t = rho * error_(t-1) + innovation_t")
    print(f"  rho = {rho:.3f}")
    print(f"  sigma = {sigma:.3f}")
    print()
    print("OLS slope estimate:")
    print(f"  estimated slope = {model.params[1]:.5f}")
    print(f"  true slope      = {beta1:.5f}")
    print()
    print("Standard errors for slope:")
    print(f"  conventional OLS SE, assumes independent errors = {conventional_se:.5f}")
    print(f"  HC3 robust SE, handles heteroskedasticity but not time dependence = {robust_hc3_se:.5f}")
    print(f"  ordinary pairs bootstrap SE, resamples points as if independent = {ordinary_boot_se:.5f}")
    print(f"  moving block bootstrap SE, preserves local dependence = {block_boot_se:.5f}")
    print()
    print("Ratios:")
    print(f"  block bootstrap / conventional OLS = {block_boot_se / conventional_se:.3f}")
    print(f"  block bootstrap / ordinary bootstrap = {block_boot_se / ordinary_boot_se:.3f}")
    print()
    print("Interpretation:")
    print("  The plotted data are one observed time-ordered sample.")
    print("  The ordinary OLS SE and ordinary bootstrap treat observations as independent.")
    print("  The block bootstrap resamples runs of nearby observations together,")
    print("  so it better reflects the loss of information caused by temporal correlation.")


interact(
    draw_demo,
    n=IntSlider(value=250, min=50, max=800, step=50, description="n"),
    beta0=FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.5, description="beta0"),
    beta1=FloatSlider(value=0.06, min=-0.5, max=0.5, step=0.01, description="beta1"),
    rho=FloatSlider(value=0.80, min=0.0, max=0.98, step=0.02, description="rho"),
    sigma=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description="sigma"),
    x_noise=FloatSlider(value=0.30, min=0.0, max=2.0, step=0.05, description="x_noise"),
    B=IntSlider(value=1000, min=200, max=5000, step=200, description="bootstrap B"),
    block_length=IntSlider(value=15, min=2, max=100, step=1, description="block length"),
    seed=IntSlider(value=7, min=1, max=999, step=1, description="seed")
)

interactive(children=(IntSlider(value=250, description='n', max=800, min=50, step=50), FloatSlider(value=2.0, …

<function __main__.draw_demo(n=250, beta0=2.0, beta1=0.06, rho=0.8, sigma=2.0, x_noise=0.3, B=1000, block_length=15, seed=7)>

## The main lesson is:

## > Correlated observations reduce the effective sample size. You may have many rows, but fewer independent pieces of information.

## Practical takeaway:

## - If data are independent, ordinary OLS standard errors may be fine.
## - If nearby observations are correlated, ordinary OLS standard errors can be misleading.
## - A block bootstrap or time-series-aware method is safer for estimating uncertainty.

# Why log transformations are common in regression

## A short motivation:

## Logs convert multiplicative relationships into additive ones and often make skewed variables easier to model.

# A concrete case study:

## Revenue, population, income, website traffic, and prices are often right-skewed. A log model can express effects in percentage terms.

# A warning:

## log(0) is undefined. If Y or X can be zero or negative, we need a modeling decision: filter, transform with log1p, use another model, or rethink the scale.

# Log Transformation

In [ ]:
Y = beta1 * X^2
Y = e^beta0 * X^beta1 * e^epsilon
e^Y = e^beta0 * X^beta1 * e^epsilon
Y = exp(beta0 + beta1 X + epsilon)

from IPython.display import display, HTML

display(HTML("""
<div style="font-size: 30px; line-height: 1.45; max-width: 1100px;">

<h2 style="font-size: 40px;">Common Log-Transformed Regression Forms</h2>

<ul>

<li>
<b>log(Y) = β<sub>0</sub> + β<sub>1</sub> log(X) + ε</b><br>
<b>Log-log model.</b> A 1% increase in X is associated with about a β<sub>1</sub>% change in Y.
</li>

<br>

<li>
<b>Y = β<sub>0</sub> + β<sub>1</sub> log(X) + ε</b><br>
<b>Linear-log model.</b> A 1% increase in X is associated with about a β<sub>1</sub>/100 unit change in Y.
</li>

<br>

<li>
<b>log(Y) = β<sub>0</sub> + β<sub>1</sub>X + ε</b><br>
<b>Log-linear model.</b> A 1-unit increase in X is associated with about a 100β<sub>1</sub>% change in Y.
</li>

</ul>

</div>
"""))

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import display, HTML

display(HTML("""
<div style="font-size: 28px; line-height: 1.45; max-width: 1100px;">

<h1 style="font-size: 42px;">Simple Visual: Why Use a Log Transformation?</h1>

<p>
A log transformation can turn some curved relationships into straight-line relationships.
This is useful because ordinary linear regression fits straight lines.
</p>

<div style="background-color:#eaf4ff; padding: 18px; margin: 14px 0;">
If <b>Y = exp(β<sub>0</sub> + β<sub>1</sub>X)</b>, then
<br><br>
<b>log(Y) = β<sub>0</sub> + β<sub>1</sub>X</b>.
</div>

<p>
So the relationship between <b>X</b> and <b>Y</b> is curved, but the relationship between
<b>X</b> and <b>log(Y)</b> is linear.
</p>

</div>
"""))

def draw_log_transform_demo(beta0=1.0, beta1=0.4):
    x = np.linspace(0, 10, 300)

    y = np.exp(beta0 + beta1 * x)
    log_y = np.log(y)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].plot(x, y, color="tab:blue", linewidth=3)
    axes[0].set_title("Original Scale: Y vs. X", fontsize=16)
    axes[0].set_xlabel("X", fontsize=13)
    axes[0].set_ylabel("Y", fontsize=13)
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, log_y, color="tab:orange", linewidth=3)
    axes[1].set_title("Log Scale: log(Y) vs. X", fontsize=16)
    axes[1].set_xlabel("X", fontsize=13)
    axes[1].set_ylabel("log(Y)", fontsize=13)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="font-size: 24px; line-height: 1.45; max-width: 1000px;">

    <div style="background-color:#d4edda; padding: 16px; margin-top: 12px;">
    In this example:
    <br><br>
    <b>Y = exp({beta0:.2f} + {beta1:.2f}X)</b>
    <br>
    so
    <br>
    <b>log(Y) = {beta0:.2f} + {beta1:.2f}X</b>
    </div>

    <div style="background-color:#fff3cd; padding: 16px; margin-top: 12px;">
    <b>Interpretation:</b> a 1-unit increase in X is associated with approximately
    <b>{100 * beta1:.1f}%</b> higher Y.
    </div>

    </div>
    """))

interact(
    draw_log_transform_demo,
    beta0=FloatSlider(
        value=1.0,
        min=-2.0,
        max=3.0,
        step=0.25,
        description="β0",
        continuous_update=False,
    ),
    beta1=FloatSlider(
        value=0.4,
        min=-0.5,
        max=1.0,
        step=0.05,
        description="β1",
        continuous_update=False,
    ),
);

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='β0', max=3.0, min=-2.0, ste…